In [1]:

!pip -q install "flask>=3.0,<4.0"

import os
import json
import time
import uuid
import hashlib
import sqlite3
import threading
import requests

from pathlib import Path
from datetime import datetime, timezone
from flask import Flask, jsonify, request
from werkzeug.serving import make_server
from google.colab import output

# 1. PROJECT SETUP
PROJECT = Path("/content/zetheta_project_1b")
(PROJECT / "src").mkdir(parents=True, exist_ok=True)
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / "tests").mkdir(parents=True, exist_ok=True)

DB = str(PROJECT / "data" / "integration.db")

# RESET OLD DATABASE TO PREVENT SCHEMA CONFLICTS

import os

if os.path.exists(DB):
    os.remove(DB)

print("Old database removed successfully!")



def now():
    return datetime.now(timezone.utc).isoformat()

def connect():
    con = sqlite3.connect(DB)
    con.row_factory = sqlite3.Row
    return con

# 2. DATABASE CREATION
with connect() as con:
    con.executescript("""
    CREATE TABLE IF NOT EXISTS journal_entries(
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        source_id TEXT UNIQUE,
        company_code TEXT,
        document_number TEXT,
        fiscal_year INTEGER,
        posting_date TEXT,
        gl_account TEXT,
        cost_center TEXT,
        currency TEXT,
        amount REAL,
        debit_credit TEXT,
        tenant_id TEXT,
        payload_hash TEXT,
        processed_at TEXT
    );

    CREATE TABLE IF NOT EXISTS dead_letter_queue(
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        source_id TEXT,
        payload TEXT,
        error TEXT,
        attempts INTEGER,
        created_at TEXT,
        resolved INTEGER DEFAULT 0
    );

    CREATE TABLE IF NOT EXISTS audit_log(
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        correlation_id TEXT,
        event TEXT,
        details TEXT,
        created_at TEXT
    );
    """)

# 3. SIMULATED SAP S/4HANA DATA
SAP_DATA = [
    {
        "source_id": "SAP-10001",
        "company_code": "IN01",
        "document_number": "190000001",
        "fiscal_year": 2026,
        "posting_date": "2026-04-02",
        "gl_account": "400100",
        "cost_center": "MFG-01",
        "currency": "INR",
        "amount": 125000,
        "debit_credit": "C",
        "fiscal_period": 1
    },
    {
        "source_id": "SAP-10002",
        "company_code": "IN01",
        "document_number": "190000002",
        "fiscal_year": 2026,
        "posting_date": "2026-04-03",
        "gl_account": "510200",
        "cost_center": "MFG-02",
        "currency": "INR",
        "amount": 48000.50,
        "debit_credit": "D",
        "fiscal_period": 1
    },
    {
        "source_id": "SAP-10003",
        "company_code": "IN02",
        "document_number": "190000003",
        "fiscal_year": 2026,
        "posting_date": "2026-04-05",
        "gl_account": "610300",
        "cost_center": "HQ-01",
        "currency": "INR",
        "amount": 22000,
        "debit_credit": "D",
        "fiscal_period": 1
    },
    {
        "source_id": "SAP-10004",
        "company_code": "IN01",
        "document_number": "190000004",
        "fiscal_year": 2026,
        "posting_date": "2026-05-01",
        "gl_account": "400100",
        "cost_center": "MFG-01",
        "currency": "INR",
        "amount": 155000,
        "debit_credit": "C",
        "fiscal_period": 2
    }
]

# 4. DATA EXTRACTION
class SAPAdapter:
    def fetch_data(self):
        return [dict(record) for record in SAP_DATA]

# 5. DATA TRANSFORMATION
def transform_data(raw):
    required = [
        "source_id", "company_code", "document_number",
        "fiscal_year", "posting_date", "gl_account",
        "currency", "amount", "debit_credit"
    ]

    for field in required:
        if raw.get(field) in (None, ""):
            raise ValueError("Missing field: " + field)

    if raw["debit_credit"] not in ("D", "C"):
        raise ValueError("Invalid debit/credit indicator")

    amount = float(raw["amount"])
    if amount < 0:
        raise ValueError("Amount cannot be negative")

    tenant_map = {
        "IN01": "FIN-INDIA-MFG",
        "IN02": "FIN-INDIA-HQ"
    }

    company = str(raw["company_code"])
    if company not in tenant_map:
        raise ValueError("Unmapped company code")

    posting_date = datetime.strptime(
        raw["posting_date"], "%Y-%m-%d"
    ).date().isoformat()

    return {
        "source_system": "SAP_S4HANA",
        "source_id": str(raw["source_id"]),
        "tenant_id": tenant_map[company],
        "company_code": company,
        "document_number": str(raw["document_number"]),
        "fiscal_year": int(raw["fiscal_year"]),
        "fiscal_period": int(raw.get("fiscal_period", 0)),
        "posting_date": posting_date,
        "calendar_month": posting_date[:7],
        "gl_account": str(raw["gl_account"]),
        "cost_center": raw.get("cost_center"),
        "currency": str(raw["currency"]).upper(),
        "amount": round(amount, 2),
        "debit_credit": raw["debit_credit"]
    }

# 6. SIMULATED FINSIGHT DESTINATION
class FinSightAdapter:
    def __init__(self):
        self.sent = []

    def send(self, record):
        self.sent.append(record)
        return {
            "accepted": True,
            "target_reference": "FS-" + record["source_id"]
        }

sap = SAPAdapter()
finsight = FinSightAdapter()

# 7. AUDIT LOGGING
def audit(correlation_id, event, details):
    with connect() as con:
        con.execute(
            """INSERT INTO audit_log
            (correlation_id,event,details,created_at)
            VALUES(?,?,?,?)""",
            (correlation_id, event, json.dumps(details), now())
        )




# STEP 8: DEAD LETTER QUEUE

import sqlite3
import json
from datetime import datetime, timezone

DB = "/content/zetheta_project_1b/data/integration.db"

def save_failure(raw, error, attempts):

    connection = sqlite3.connect(DB)

    cursor = connection.cursor()

    cursor.execute(
        "CREATE TABLE IF NOT EXISTS dead_letter_queue ("
        "id INTEGER PRIMARY KEY AUTOINCREMENT, "
        "source_id TEXT, "
        "payload TEXT, "
        "error TEXT, "
        "attempts INTEGER, "
        "created_at TEXT, "
        "resolved INTEGER DEFAULT 0)"
    )

    cursor.execute(
        "INSERT INTO dead_letter_queue "
        "(source_id, payload, error, attempts, created_at) "
        "VALUES (?, ?, ?, ?, ?)",
        (
            raw.get("source_id"),
            json.dumps(raw),
            str(error),
            attempts,
            datetime.now(timezone.utc).isoformat()
        )
    )

    connection.commit()
    connection.close()

print("Dead Letter Queue created successfully!")


# 9. INTEGRATION PROCESSING
def process_record(raw, correlation_id=None):
    correlation_id = correlation_id or str(uuid.uuid4())

    try:
        record = transform_data(raw)

        source_id = record["source_id"]
        digest = hashlib.sha256(
            json.dumps(record, sort_keys=True).encode()
        ).hexdigest()

        with connect() as con:
            existing = con.execute(
                "SELECT id FROM journal_entries WHERE source_id=?",
                (source_id,)
            ).fetchone()

        if existing:
            audit(correlation_id, "duplicate_skipped", {
                "source_id": source_id
            })
            return {
                "source_id": source_id,
                "status": "duplicate_skipped"
            }

        response = None
        last_error = None

        for attempt in range(1, 4):
            try:
                response = finsight.send(record)
                break
            except Exception as error:
                last_error = str(error)
                time.sleep(0.1 * attempt)

        if response is None:
            save_failure(raw, last_error, 3)
            audit(correlation_id, "sent_to_dlq", {
                "source_id": source_id,
                "error": last_error
            })
            return {
                "source_id": source_id,
                "status": "dead_lettered",
                "error": last_error
            }

        with connect() as con:
            con.execute(
                """INSERT INTO journal_entries
                (source_id,company_code,document_number,fiscal_year,
                posting_date,gl_account,cost_center,currency,amount,
                debit_credit,tenant_id,payload_hash,processed_at)
                VALUES(?,?,?,?,?,?,?,?,?,?,?,?,?)""",
                (
                    source_id,
                    record["company_code"],
                    record["document_number"],
                    record["fiscal_year"],
                    record["posting_date"],
                    record["gl_account"],
                    record["cost_center"],
                    record["currency"],
                    record["amount"],
                    record["debit_credit"],
                    record["tenant_id"],
                    digest,
                    now()
                )
            )

        audit(correlation_id, "record_delivered", {
            "source_id": source_id,
            "target": response
        })

        return {
            "source_id": source_id,
            "status": "delivered",
            "target": response
        }

    except Exception as error:
        save_failure(raw, str(error), 0)
        audit(correlation_id, "validation_failed", {
            "source_id": raw.get("source_id"),
            "error": str(error)
        })
        return {
            "source_id": raw.get("source_id"),
            "status": "rejected",
            "error": str(error)
        }

# 10. COMPLETE PIPELINE
def run_pipeline():
    correlation_id = str(uuid.uuid4())
    records = sap.fetch_data()

    results = [
        process_record(record, correlation_id)
        for record in records
    ]

    audit(correlation_id, "batch_completed", {
        "count": len(records),
        "results": results
    })

    return {
        "correlation_id": correlation_id,
        "fetched": len(records),
        "results": results
    }

# 11. METRICS
def get_metrics():
    with connect() as con:
        delivered = con.execute(
            "SELECT COUNT(*) FROM journal_entries"
        ).fetchone()[0]

        pending = con.execute(
            "SELECT COUNT(*) FROM dead_letter_queue WHERE resolved=0"
        ).fetchone()[0]

        audit_count = con.execute(
            "SELECT COUNT(*) FROM audit_log"
        ).fetchone()[0]

    return {
        "records_delivered": delivered,
        "dead_letter_pending": pending,
        "audit_events": audit_count
    }

# 12. FLASK REST API
app = Flask(__name__)

@app.get("/")
def home():
    return jsonify({
        "project": "Zetheta 1B",
        "name": "SAP S/4HANA to FinSight Integration",
        "status": "running",
        "endpoints": [
            "/health",
            "/api/v1/sync",
            "/api/v1/journal-entries",
            "/api/v1/metrics",
            "/api/v1/dead-letter-queue"
        ]
    })

@app.get("/health")
def health():
    return jsonify({
        "status": "healthy",
        "service": "SAP-FinSight Integration"
    })

@app.post("/api/v1/sync")
def sync():
    return jsonify(run_pipeline())

@app.get("/api/v1/journal-entries")
def journal_entries():
    limit = min(max(request.args.get("limit", 50, type=int), 1), 200)

    with connect() as con:
        rows = con.execute(
            "SELECT * FROM journal_entries ORDER BY id DESC LIMIT ?",
            (limit,)
        ).fetchall()

    return jsonify([dict(row) for row in rows])

@app.get("/api/v1/metrics")
def metrics():
    return jsonify(get_metrics())

@app.get("/api/v1/dead-letter-queue")
def dead_letter_queue():
    with connect() as con:
        rows = con.execute(
            "SELECT * FROM dead_letter_queue ORDER BY id DESC"
        ).fetchall()

    return jsonify([dict(row) for row in rows])

# 13. AUTOMATIC TESTING
print("\n" + "=" * 55)
print("ZETHETA 1B - AUTOMATED PROJECT TESTING")
print("=" * 55)

client = app.test_client()

def test(name, method, url, expected=200):
    response = getattr(client, method)(url)
    passed = response.status_code == expected
    print(("PASS" if passed else "FAIL"), "-", name,
          "| Status:", response.status_code)
    if not passed:
        print(response.get_data(as_text=True))
    return passed

results = []

results.append(test("Homepage", "get", "/"))
results.append(test("Health Check", "get", "/health"))

# Run integration pipeline
response = client.post("/api/v1/sync")
print("\nINTEGRATION RESULTS:")
print(json.dumps(response.get_json(), indent=2))

results.append(response.status_code == 200)

results.append(test(
    "Journal Entries API", "get", "/api/v1/journal-entries"
))
results.append(test("Metrics API", "get", "/api/v1/metrics"))
results.append(test(
    "Dead Letter Queue API", "get", "/api/v1/dead-letter-queue"
))

# Duplicate prevention test
duplicate_result = process_record(SAP_DATA[0])
print("\nDUPLICATE TEST:", duplicate_result)
results.append(duplicate_result["status"] == "duplicate_skipped")

# Invalid data validation test
invalid = dict(SAP_DATA[0])
invalid["source_id"] = "SAP-INVALID-TEST"
invalid["debit_credit"] = "X"
invalid_result = process_record(invalid)
print("\nVALIDATION TEST:", invalid_result)
results.append(invalid_result["status"] == "rejected")

# 14. SAVE PROJECT SOURCE FILES
Path(PROJECT / "requirements.txt").write_text(
    "Flask>=3.0,<4.0\n"
)

Path(PROJECT / "src" / "integration.py").write_text(
    "# Main integration logic is implemented in this notebook cell.\n"
)

Path(PROJECT / "src" / "api.py").write_text(
    "# Flask API routes are implemented in this notebook cell.\n"
)

# 15. START WEB SERVER
server = None
PORT = 8000

try:
    server = make_server("0.0.0.0", PORT, app)
except OSError:
    PORT = 8001
    server = make_server("0.0.0.0", PORT, app)

thread = threading.Thread(target=server.serve_forever, daemon=True)
thread.start()

time.sleep(1)

print("\n" + "=" * 55)
print("FINAL PROJECT SUMMARY")
print("=" * 55)

print("Project Folder:", PROJECT)
print("Database:", DB)
print("Total Tests:", len(results))
print("Passed Tests:", sum(results))
print("Failed Tests:", len(results) - sum(results))
print("Metrics:", json.dumps(get_metrics(), indent=2))

print("\nPROJECT API LINK:")
print(output.eval_js(
    f'google.colab.kernel.proxyPort({PORT})'
))

print("\nPROJECT STATUS:",
      "SUCCESS" if all(results) else "CHECK FAILED TESTS")

print("\nZETHETA 1B EXECUTION COMPLETED!")


Old database removed successfully!
Dead Letter Queue created successfully!

ZETHETA 1B - AUTOMATED PROJECT TESTING
PASS - Homepage | Status: 200
PASS - Health Check | Status: 200

INTEGRATION RESULTS:
{
  "correlation_id": "fd649985-7012-45aa-8e99-aa085f0a09ab",
  "fetched": 4,
  "results": [
    {
      "source_id": "SAP-10001",
      "status": "delivered",
      "target": {
        "accepted": true,
        "target_reference": "FS-SAP-10001"
      }
    },
    {
      "source_id": "SAP-10002",
      "status": "delivered",
      "target": {
        "accepted": true,
        "target_reference": "FS-SAP-10002"
      }
    },
    {
      "source_id": "SAP-10003",
      "status": "delivered",
      "target": {
        "accepted": true,
        "target_reference": "FS-SAP-10003"
      }
    },
    {
      "source_id": "SAP-10004",
      "status": "delivered",
      "target": {
        "accepted": true,
        "target_reference": "FS-SAP-10004"
      }
    }
  ]
}
PASS - Journal Entries AP